# R6 · TwinCheck demo (Rathi)

**Runtime → T4 GPU** (or better), switch on **Notebook access** for `GITHUB_TOKEN` and `HF_TOKEN` in the 🔑 panel, then **Run all**.

Steps: 1 setup → 2 tests → 3 run the 6 examples and push the results (ends with `PUSHED`) → 4 launch the app (prints a public `gradio.live` link; this last cell keeps running while the demo is open, so stop it when you are done).

Uses Himasri's trained verdict head (`models/verdict_twin_s42`) and calibration; the status line in step 1 should show every component as `real`.

In [ ]:
# 1. Setup
from google.colab import userdata
import os, json
REPO = '/content/twincheck'
if not os.path.exists(REPO):
    !git clone https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
!git fetch -q origin && git checkout -q rathi/R6-demo && git pull -q origin rathi/R6-demo

GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
!pip install -q -U gradio bitsandbytes rank_bm25 faiss-cpu
!python -m spacy download -q en_core_web_trf   # entities for the verdict head
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
from huggingface_hub import HfApi
USER = HfApi(token=os.environ['HF_TOKEN']).whoami()['name']
os.environ['TWINCHECK_NLI'] = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'   # zero-shot until R4 v2 wins
!python -m src.retrieval.build_index download --repo {USER}/twincheck-index-v1
!python -m src.pipeline --status

## 2. Tests

In [ ]:
!python -m pytest tests/demo tests/test_pipeline.py -q

## 3. Run the 6 ready examples with the real models and push what the model says

In [ ]:
!python -m src.demo.app --run-examples results/R6/examples.json
# PUSHED is printed only if the examples ran, and the commit and push worked
!git add results/R6/examples.json && git commit -q -m "[R6] Demo example results with trained verdict head" && git push -q origin rathi/R6-demo && echo PUSHED

## 4. Launch the demo (click the `https://….gradio.live` link). Stop the cell to close it.

In [ ]:
!python -m src.demo.app --share